# Figure 5: lipid-screen MGMs connect CRISPR-validated genes to disease

Exported at **180 x 134 mm**.

- **a**: the four MGMs selected from the ten CLAMP-full canonical ARCHS4 models (`workflow/rules/archs4_crispercas.smk`, rule `crispercas_metalv_select`): CRISPR screen genes (diamonds) and the top 10 genes by loading, edges for genes sharing an enriched pathway, pathway and cell-type labels, the sample types of the top 1% ARCHS4 samples by MGM activity, and links to the diseases merged from the PhenoPLIER GLS traits (FDR < 0.05, at least 1,000 cases). Circled genes have S-MultiXcan FDR < 0.05 for the linked disease.
- **b**: S-MultiXcan P values and fastENLOC RCP (PhenomeXcan) of the screen genes and reference genes for the Figure 5 traits; diamonds mark each screen gene with the traits of its MGM.

Source tables: `output/03_model_biology/02_archs4/04_crispercas/00_mgm/fig5/` (rule `crispercas_fig5_data`).

💡 **Environment:** `clamp-analyses` (R kernel)

In [ ]:
fread_mgm <- function(f) {
  d <- fread(f)
  if ("metaLV" %in% names(d)) d[, metaLV := sub("^metaLV", "MGM", metaLV)]
  d
}
source(here::here("scripts/panels/main_figure.R"))
suppressPackageStartupMessages(library(ggforce))
dirs <- main_figure_dirs("fig5")
D <- here("output/03_model_biology/02_archs4/04_crispercas/00_mgm/fig5")
for (f in list.files(D, pattern = "\\.csv$", full.names = TRUE)) file.copy(f, file.path(dirs$source, basename(f)), overwrite = TRUE)
FIG_W <- 180; FIG_H <- 134
PANEL_W <- 122; PANEL_H <- 123

mods <- fread_mgm(file.path(D, "modules.csv"))
genes <- fread_mgm(file.path(D, "module_genes.csv"))
pedges <- fread_mgm(file.path(D, "pathway_edges.csv"))
mt <- fread_mgm(file.path(D, "module_trait_groups.csv"))
gedges <- fread_mgm(file.path(D, "gene_trait_edges.csv"))
rp <- fread_mgm(file.path(D, "ring_pvalues.csv"))[as.numeric(fdr) < 0.05]
gedges <- gedges[paste(metaLV, gene, trait_group) %in% rp[, paste(metaLV, gene, disease)]]
tw <- fread_mgm(file.path(D, "twas_rcp_matrix.csv"))
proj <- fread_mgm(file.path(D, "projection_composition.csv"))

dname <- c(Gallstones = "Gallstone disease", "Alzheimer's" = "Alzheimer's disease", "High cholesterol" = "Hypercholesterolemia", "Venous thrombosis" = "Venous thromboembolism")
mt[, disease := dname[trait_group]]; gedges[, disease := dname[trait_group]]
mt <- mt[!is.na(disease)]; gedges <- gedges[!is.na(disease)]
pcol0 <- c("Gallstone disease" = "#B07A2A", "Alzheimer's disease" = "#8870AA", "Hypercholesterolemia" = "#378A80", "Venous thromboembolism" = "#B5485E")
mix <- function(a, b) colorRampPalette(c(a, b))(3)[2]
ncol_ <- c(MGM175 = pcol0[[1]], MGM335 = pcol0[[2]], MGM588 = pcol0[[3]], MGM91 = pcol0[[4]])
pcol <- c("Gallstone disease" = "#B07A2A", "Alzheimer's disease" = "#8870AA", "Hypercholesterolemia" = "#378A80", "Venous thromboembolism" = "#B5485E")
ccol <- c("liver / hepatocyte" = "#4E79A7", "monocyte / macrophage" = "#9C755F", "pancreas / islet / ductal" = "#EDC948",
          "blood / immune" = "#FF9DA7", "brain / neural" = "#59A14F", "tumour / cancer" = "#5F5F5F",
          "cell-free / plasma" = "#BAB0AC", "other" = "#E6E6E6")

memb <- genes[, .(group = if (.N > 1) paste(sort(metaLV), collapse = "+") else metaLV[1], crispr = any(crispr), rank = min(rank)), by = gene]
arc <- function(g, cx, cy, r, a0, a1, ry = r) {
  n <- length(g); a <- if (n == 1) (a0 + a1) / 2 else seq(a0, a1, length.out = n)
  data.table(gene = g, x = cx + r * cos(a), y = cy + ry * sin(a))
}
ord <- function(grp) memb[group == grp][order(-crispr, rank), gene]
pos <- rbind(
  arc(ord("MGM175"), 21, 86, 10, pi * 0.95, pi * 0.95 - 2 * pi * (9 / 11)),
  data.table(gene = "DGAT2", x = 25, y = 66),
  arc(ord("MGM335"), 22, 46, 10, pi * 0.95, pi * 0.95 - 2 * pi * (9 / 11)),
  arc(ord("MGM588"), 78, 87, 10.5, pi * 1.12, -pi * 0.12),
  data.table(gene = c("HMGCS1", "ACACA", "LDLR"), x = c(74, 82, 90), y = 66.5),
  arc(c("STARD4", "ACSS2", "MVD", "MVK", "FADS2", "HSD17B7", "INSIG1", "PCSK9"), 79, 45, 10.5, pi * 0.88, pi * 2.12)
)
pos[gene == "INSIG1", x := x + 7]
pos[gene == "HSD17B7", `:=`(x = x + 2, y = y - 2)]
pos <- merge(pos, memb, by = "gene")
cxy <- pos[, .(cx = mean(x), cy = mean(y)), by = group]
pos <- merge(pos, cxy, by = "group")
pos[, `:=`(ang = atan2(y - cy, x - cx))]
pos[, `:=`(tx = x + 2.5 * cos(ang), ty = y + 2.5 * sin(ang),
           hj = ifelse(cos(ang) > .3, 0, ifelse(cos(ang) < -.3, 1, .5)), vj = ifelse(abs(cos(ang)) > .3, .5, ifelse(sin(ang) > 0, 0, 1)))]
pos[gene == "ASGR2", ty := ty + 1.3]
pos[gene == "DGAT2", `:=`(tx = x + 2.6, ty = y, hj = 0, vj = .5)]
pos[gene == "ACACA", `:=`(tx = x, ty = y + 3.3, hj = .5, vj = 0)]
pos[gene %in% c("LDLR", "HMGCS1"), `:=`(tx = x, ty = y - 2.6, hj = .5, vj = 1)]
pos[gene == "ACACA", `:=`(tx = x, ty = y - 2.6, hj = .5, vj = 1)]
pos[, `:=`(lw = nchar(gene) * .95 + .8, lh = 2.3)]
pos[, `:=`(lx0 = tx - hj * lw, ly0 = ty - vj * lh)]

CIRC <- list()
hull <- rbindlist(lapply(c("MGM175", "MGM335", "MGM588", "MGM91"), function(m) {
  p <- pos[gene %in% genes[metaLV == m, gene]]
  a <- seq(0, 2 * pi, length.out = 25)[-25]
  pts <- rbind(p[, .(x, y)], p[, .(x = c(lx0, lx0 + lw, lx0, lx0 + lw), y = c(ly0, ly0, ly0 + lh, ly0 + lh)), by = gene][, .(x, y)])
  ctr <- c(mean(range(pts$x)), mean(range(pts$y)))
  e <- eigen(cov(cbind(pts$x, pts$y)))$vectors
  u <- (pts$x - ctr[1]) * e[1, 1] + (pts$y - ctr[2]) * e[2, 1]
  v <- (pts$x - ctr[1]) * e[1, 2] + (pts$y - ctr[2]) * e[2, 2]
  aa <- max(abs(u)) + 1; bb <- max(abs(v)) + 1
  sc <- max(sqrt((u / aa)^2 + (v / bb)^2)) * 1.04
  aa <- aa * sc; bb <- bb * sc
  t <- seq(0, 2 * pi, length.out = 181)[-181]
  sx <- ctr[1] + aa * cos(t) * e[1, 1] + bb * sin(t) * e[1, 2]
  sy <- ctr[2] + aa * cos(t) * e[2, 1] + bb * sin(t) * e[2, 2]
  if (m %in% names(CIRC)) {
    cc <- CIRC[[m]]; t <- seq(0, 2 * pi, length.out = 181)[-181]
    sx <- cc[1] + cc[3] * cos(t); sy <- cc[2] + cc[4] * sin(t)
  }
  data.table(metaLV = m, x = sx, y = sy)
}))
DX <- 57
hull <- rbind(hull[metaLV %in% c("MGM175", "MGM335")],
              hull[metaLV == "MGM175"][, `:=`(metaLV = "MGM588", x = x + DX)],
              hull[metaLV == "MGM335"][, `:=`(metaLV = "MGM91", x = x + DX, y = y + 3)])

dis <- data.table(disease = names(pcol), x = c(49, 51, 111.5, 49), y = c(103, 64, 66, 30), w = c(14, 14, 21, 18.5))
hubs <- merge(mt[, .(metaLV, disease, fdr)], dis, by = "disease")
cen <- pos[, .(mx = mean(x), my = mean(y)), by = .(metaLV = group)]
hubs <- merge(hubs, cen, by = "metaLV")
mc <- rbindlist(lapply(unique(hubs$metaLV), function(m) pos[gene %in% genes[metaLV == m, gene], .(metaLV = m, mx = mean(x), my = mean(y))][1]))
hubs <- merge(hubs[, !c("mx", "my")], mc, by = "metaLV")
for (i in seq_len(nrow(hubs))) {
  h <- hull[metaLV == hubs$metaLV[i]]
  tgt <- atan2(hubs$y[i] - hubs$my[i], hubs$x[i] - hubs$mx[i])
  da <- abs(atan2(sin(atan2(h$y - hubs$my[i], h$x - hubs$mx[i]) - tgt), cos(atan2(h$y - hubs$my[i], h$x - hubs$mx[i]) - tgt)))
  j <- which.min(da)
  if (hubs$disease[i] == "Hypercholesterolemia") {
    ty <- hubs$y[i] - 2; ok <- which(h$x > hubs$mx[i])
    j <- ok[which.min(abs(h$y[ok] - ty))]
  }
  set(hubs, i, c("hx", "hy"), list(h$x[j], h$y[j]))
}
shorten <- function(x, y, xe, ye, b) {
  l <- sqrt((xe - x)^2 + (ye - y)^2); list(xe - (xe - x) * b / l, ye - (ye - y) * b / l)
}
FORCE <- data.table(gene = "LDLR", disease = "Hypercholesterolemia")
gfree <- gedges[!is.na(disease)][!FORCE, on = c("gene", "disease")]
few <- gfree[, .N, by = .(metaLV, disease)][N <= 2]
direct <- unique(rbind(merge(few, gfree[, .(metaLV, disease, gene)], by = c("metaLV", "disease"))[, .(gene, disease)], FORCE))
direct <- merge(merge(direct, pos[, .(gene, hx = x, hy = y, crispr)], by = "gene"), dis[, .(disease, x, y, w)], by = "disease")
GW <- 1
direct <- merge(direct, gedges[, .(p = min(pvalue)), by = .(gene, disease)], by = c("gene", "disease"))
direct[, lt := ifelse(p < GW, "solid", "22")]
hubs <- hubs[!few, on = c("metaLV", "disease")]
nogene <- mt[!paste(metaLV, disease) %in% gedges[, paste(metaLV, disease)], .(metaLV, disease)]
hubs <- hubs[paste(metaLV, disease) %in% c(gfree[!direct, on = c("gene", "disease"), paste(metaLV, disease)], nogene[, paste(metaLV, disease)])]
hubs[, lt := ifelse(paste(metaLV, disease) %in% nogene[, paste(metaLV, disease)], "22", "solid")]
hubs <- hubs[lt == "solid"]
inpoly <- function(px, py, P) { n <- nrow(P); j <- n; ins <- FALSE
  for (i in 1:n) { if (((P$y[i] > py) != (P$y[j] > py)) && (px < (P$x[j] - P$x[i]) * (py - P$y[i]) / (P$y[j] - P$y[i]) + P$x[i])) ins <- !ins; j <- i }
  ins }
for (d in unique(hubs[lt == "22", disease])) {
  ms <- hubs[lt == "22" & disease == d, metaLV]
  if (length(ms) == 2) {
    A <- hull[metaLV == ms[1]]; B <- hull[metaLV == ms[2]]
    cand <- rbind(A[vapply(seq_len(nrow(A)), function(i) inpoly(A$x[i], A$y[i], B), logical(1))],
                  B[vapply(seq_len(nrow(B)), function(i) inpoly(B$x[i], B$y[i], A), logical(1))])
    k <- which.min(abs(cand$x - hubs[lt == "22" & disease == d, x][1]) + abs(cand$y - hubs[lt == "22" & disease == d, y][1]))
    hubs[lt == "22" & disease == d, `:=`(hx = cand$x[k], hy = cand$y[k])]
    hubs <- hubs[!(lt == "22" & disease == d & metaLV == ms[2])]
  }
}
gedges_hub <- gfree[!few, on = c("metaLV", "disease")]
gedges_hub <- gedges_hub[!direct, on = c("gene", "disease")]
hubs[, c("ex", "ey") := shorten(hx, hy, x, y, 7.6)]

ge <- merge(merge(gedges_hub[, .(metaLV, gene, disease, p = pvalue)], pos[, .(gene, x, y, crispr)], by = "gene"),
            hubs[, .(metaLV, disease, hx, hy)], by = c("metaLV", "disease"))
ge <- ge[!(abs(x - hx) < 1e-6 & abs(y - hy) < 1e-6)]
ge[, `:=`(gl = sqrt((hx - x)^2 + (hy - y)^2), r0 = ifelse(crispr, 2.5, 1.95))]
ge[, `:=`(x = x + (hx - x) * r0 / gl, y = y + (hy - y) * r0 / gl)]
pe <- merge(pedges, pos[, .(gene1 = gene, x, y)], by = "gene1")
pe <- unique(merge(pe, pos[, .(gene2 = gene, xend = x, yend = y)], by = "gene2"), by = c("gene1", "gene2"))
ring <- gedges[order(pvalue)][, .SD[1], by = gene]
pos <- merge(pos, ring[, .(gene, ring = disease, ring_lt = ifelse(pvalue < GW, "solid", "22"))], by = "gene", all.x = TRUE)

short <- c("liver / hepatocyte" = "liver", "monocyte / macrophage" = "macrophage", "pancreas / islet / ductal" = "pancreas",
           "blood / immune" = "blood", "brain / neural" = "brain", "tumour / cancer" = "tumour", "cell-free / plasma" = "plasma")
lab <- data.table(metaLV = c("MGM175", "MGM335", "MGM588", "MGM91"), lx = c(1, 1, 74, 72), ly = c(123, 18, 123, 18))
lab <- merge(merge(lab, mods[, .(metaLV, models, pathways, cells, drug)], by = "metaLV"),
             proj[set == "top 1%" & category != "other" & pct >= 15][order(-pct)][, .(act = paste(paste0(short[category], " ", round(pct), "%"), collapse = ", ")), by = metaLV], by = "metaLV")
lab[, title := paste0("bold('", metaLV, "')~'(", models, " models)'")]
KX <- 11; LH <- 2.15
rows <- lab[, {
  pw <- strsplit(pathways, "; ")[[1]]
  data.table(k = c("Pathways", rep("", length(pw) - 1), "Cell type", "L1000 drug", "Active in"),
             val = c(pw, sub(";.*", "", cells), drug, ""), i = seq_len(length(pw) + 3))
}, by = .(metaLV, lx, ly)]
rows[, y := ly - 3.2 - (i - 1) * LH]
BW <- 14; BH <- 1.4
bars <- merge(proj[set == "top 1%"], rows[k == "Active in", .(metaLV, lx, by1 = y)], by = "metaLV")[order(metaLV, category == "other", -pct)]
bars[, `:=`(xmax = lx + KX + BW * cumsum(pct) / 100, xmin = lx + KX + BW * (cumsum(pct) - pct) / 100), by = metaLV]
bars[, category := factor(category, levels = names(ccol))]
blab <- merge(unique(bars[, .(metaLV, lx, by1)]), lab[, .(metaLV, act)], by = "metaLV")
blk <- rows[, .(xmin = lx[1] - 1.2, ymin = min(y) - 1.9, ymax = ly[1] + 1.3,
                xmax = lx[1] + max(KX + nchar(val) * .93, 24) + 1.6), by = metaLV]
blk <- merge(blk, blab[, .(metaLV, xa = lx + KX + BW + 1.6 + nchar(act) * .93)], by = "metaLV")
blk[, xmax := pmax(xmax, xa + 1.2)]
blkp <- blk[, .(x = c(xmin, xmax, xmax, xmin), y = c(ymin, ymin, ymax, ymax)), by = metaLV]
con <- rbindlist(lapply(blk$metaLV, function(m) {
  b <- blk[metaLV == m]
  cand <- data.table(x = c((b$xmin + b$xmax) / 2, (b$xmin + b$xmax) / 2, b$xmin, b$xmax),
                     y = c(b$ymin, b$ymax, (b$ymin + b$ymax) / 2, (b$ymin + b$ymax) / 2))
  h <- hull[metaLV == m]
  d <- outer(cand$x, h$x, "-")^2 + outer(cand$y, h$y, "-")^2
  k <- which(d == min(d), arr.ind = TRUE)[1, ]
  data.table(metaLV = m, x = cand$x[k[1]], y = cand$y[k[1]], xend = h$x[k[2]], yend = h$y[k[2]])
}))

people <- dis[, .(x = x + c(-2.6, 0, 2.6), y = y + 3.2, s = c(.8, 1, .8)), by = disease]
pq <- mt[, .(q = paste0(metaLV, " FDR ", formatC(fdr, format = "e", digits = 0), collapse = "\n")), by = disease]
dis <- merge(dis, pq, by = "disease")
dis[, b := 6.2 + 1.2 * lengths(strsplit(q, "\n"))]
hubs[dis, `:=`(b = i.b, w = i.w), on = "disease"]
clamp <- function(v, lo, hi) pmin(pmax(v, lo), hi)
hubs[, side := fifelse(hx < x - w / 2, "l", fifelse(hx > x + w / 2, "r", fifelse(hy < y, "b", "t")))]
hubs[, `:=`(bx = fifelse(side %in% c("b", "t"), clamp(hx, x - w / 2 + 2.5, x + w / 2 - 2.5), fifelse(side == "l", x - w / 2, x + w / 2)),
            byy = fifelse(side == "b", y - 3.3, fifelse(side == "t", y + 3.3, clamp(hy, y - 1.8, y + 1.8))))]
hubs[, c("ex", "ey") := shorten(hx, hy, bx, byy, .45)]
direct[, side := fifelse(hx < x - w / 2, "l", fifelse(hx > x + w / 2, "r", fifelse(hy < y, "b", "t")))]
direct[, `:=`(bx = fifelse(side %in% c("b", "t"), clamp(hx, x - w / 2 + 2.5, x + w / 2 - 2.5), fifelse(side == "l", x - w / 2, x + w / 2)),
            byy = fifelse(side == "b", y - 3.3, fifelse(side == "t", y + 3.3, clamp(hy, y - 1.8, y + 1.8))))]
direct[side %in% c("l", "r"), byy := if (.N > 1) y + seq(1.8, -1.8, length.out = .N)[frank(-hy, ties.method = "first")] else byy, by = .(disease, side)]
direct[side %in% c("t", "b"), bx := if (.N > 1) x + seq(-w[1] / 2 + 3, w[1] / 2 - 3, length.out = .N)[frank(hx, ties.method = "first")] else bx, by = .(disease, side)]
direct[, c("ex", "ey") := shorten(hx, hy, bx, byy, .45)]
arr <- rbind(hubs[, .(src = "h", k = .I, disease, side, hx, hy, x, y, w)], direct[, .(src = "d", k = .I, disease, side, hx, hy, x, y, w)])
arr[side %in% c("l", "r"), `:=`(ax = ifelse(side == "l", x - w / 2, x + w / 2),
     ay = if (.N > 1) y + seq(2, -2, length.out = .N)[frank(-hy, ties.method = "first")] else clamp(hy, y - 1.8, y + 1.8)), by = .(disease, side)]
arr[side %in% c("t", "b"), `:=`(ay = ifelse(side == "b", y - 3.3, y + 3.3),
     ax = if (.N > 1) x + seq(-w[1] / 2 + 3, w[1] / 2 - 3, length.out = .N)[frank(hx, ties.method = "first")] else clamp(hx, x - w / 2 + 2.5, x + w / 2 - 2.5)), by = .(disease, side)]
hubs[arr[src == "h"]$k, `:=`(bx = arr[src == "h"]$ax, byy = arr[src == "h"]$ay)]
direct[arr[src == "d"]$k, `:=`(bx = arr[src == "d"]$ax, byy = arr[src == "d"]$ay)]
direct[gene == "FADS2" & disease == "Hypercholesterolemia", `:=`(bx = x + 3, byy = y - 3.3)]
hubs[, c("ex", "ey") := shorten(hx, hy, bx, byy, .45)]
direct[, c("ex", "ey") := shorten(hx, hy, bx, byy, .45)]
direct[, `:=`(gl = sqrt((ex - hx)^2 + (ey - hy)^2), r0 = ifelse(crispr, 2.5, 1.95))]
direct[, `:=`(sx = hx + (ex - hx) * r0 / gl, sy = hy + (ey - hy) * r0 / gl)]

up <- pos[, .(cyy = mean(y)), by = group][!grepl("\\+", group)]
glyph <- function(g, x, y, grp, crispr) {
  mods <- strsplit(grp, "\\+")[[1]]
  if (length(mods) == 2) mods <- mods[order(-up[match(mods, up$group), cyy])]
  if (crispr) {
    r <- 1.75
    full <- data.table(x = x + r * c(0, 1, 0, -1), y = y + r * c(1, 0, -1, 0))
    if (length(mods) == 1) return(full[, `:=`(gene = g, part = 1, f = mods)])
    return(rbind(data.table(x = x + r * c(-1, 0, 1), y = y + c(0, r, 0), gene = g, part = 1, f = mods[1]),
                 data.table(x = x + r * c(-1, 0, 1), y = y + c(0, -r, 0), gene = g, part = 2, f = mods[2])))
  }
  r <- 1.15
  if (length(mods) == 1) {a <- seq(0, 2 * pi, length.out = 41); return(data.table(x = x + r * cos(a), y = y + r * sin(a), gene = g, part = 1, f = mods))}
  a1 <- seq(0, pi, length.out = 21); a2 <- seq(pi, 2 * pi, length.out = 21)
  rbind(data.table(x = x + r * cos(a1), y = y + r * sin(a1), gene = g, part = 1, f = mods[1]),
        data.table(x = x + r * cos(a2), y = y + r * sin(a2), gene = g, part = 2, f = mods[2]))
}
gl <- rbindlist(lapply(seq_len(nrow(pos)), function(i) glyph(pos$gene[i], pos$x[i], pos$y[i], pos$group[i], pos$crispr[i])))
gl[, id := paste(gene, part)]
outl <- rbindlist(lapply(seq_len(nrow(pos)), function(i) {
  if (pos$crispr[i]) data.table(gene = pos$gene[i], x = pos$x[i] + 1.75 * c(0, 1, 0, -1), y = pos$y[i] + 1.75 * c(1, 0, -1, 0))
  else {a <- seq(0, 2 * pi, length.out = 41); data.table(gene = pos$gene[i], x = pos$x[i] + 1.15 * cos(a), y = pos$y[i] + 1.15 * sin(a))}
}))

segs <- rbind(ge[, .(x0 = x, y0 = y, x1 = hx, y1 = hy)], hubs[, .(x0 = hx, y0 = hy, x1 = ex, y1 = ey)],
              direct[, .(x0 = sx, y0 = sy, x1 = ex, y1 = ey)])
hit <- function(sg, xa, xb, ya, yb) {
  sum(vapply(seq_len(nrow(sg)), function(i) {
    p <- c(sg$x0[i], sg$y0[i]); d <- c(sg$x1[i] - sg$x0[i], sg$y1[i] - sg$y0[i]); t0 <- 0; t1 <- 1
    for (q in list(c(-d[1], p[1] - xa), c(d[1], xb - p[1]), c(-d[2], p[2] - ya), c(d[2], yb - p[2]))) {
      if (q[1] == 0) { if (q[2] < 0) return(FALSE) } else { r <- q[2] / q[1]; if (q[1] < 0) t0 <- max(t0, r) else t1 <- min(t1, r) }
    }
    t0 <= t1
  }, logical(1)))
}
placed <- list()
for (i in order(-pos$crispr)) {
  g <- pos[i]; lw <- nchar(g$gene) * .95 + .8; lh <- 2.3
  ang <- c(atan2(g$y - g$cy, g$x - g$cx), seq(0, 2 * pi, length.out = 9)[-9])
  best <- NULL
  for (j in seq_along(ang)) {
    a <- ang[j]; d <- ifelse(g$crispr, 3, 2.5)
    hj <- ifelse(cos(a) > .3, 0, ifelse(cos(a) < -.3, 1, .5)); vj <- ifelse(abs(cos(a)) > .3, .5, ifelse(sin(a) > 0, 0, 1))
    tx <- g$x + d * cos(a); ty <- g$y + d * sin(a); xa <- tx - hj * lw; ya <- ty - vj * lh
    sc <- hit(segs, xa, xa + lw, ya, ya + lh) * 3 + sum(pos$x > xa - 1 & pos$x < xa + lw + 1 & pos$y > ya - 1 & pos$y < ya + lh + 1) * 4 +
      sum(vapply(placed, function(b) xa < b[2] & xa + lw > b[1] & ya < b[4] & ya + lh > b[3], logical(1))) * 4 + (j > 1) * .5
    if (is.null(best) || sc < best$sc) best <- list(sc = sc, tx = tx, ty = ty, hj = hj, vj = vj, b = c(xa, xa + lw, ya, ya + lh))
  }
  set(pos, i, c("tx", "ty", "hj", "vj"), list(best$tx, best$ty, best$hj, best$vj)); placed[[length(placed) + 1]] <- best$b
}

pos[gene == "MVD", `:=`(tx = x - 2.2, ty = y - 1.6, hj = 1, vj = .5)]
pos[gene == "MVK", `:=`(tx = x - .8, ty = y - 2.9, hj = .5, vj = 1)]
pos[gene == "FADS2", `:=`(tx = x - .3, ty = y - 2.9, hj = .5, vj = 1)]
pos[gene == "INSIG1", `:=`(tx = x + 2.5, ty = y, hj = 0, vj = .5)]

## Panel a: MGM networks and linked diseases

In [ ]:
plot_a <- ggplot() +
  geom_polygon(data = hull, aes(x, y, group = metaLV, fill = metaLV), alpha = .2, colour = NA) +
  geom_segment(data = pe, aes(x, y, xend = xend, yend = yend), colour = "grey68", linewidth = .12, linetype = "dashed") +
  geom_segment(data = ge, aes(x, y, xend = hx, yend = hy, colour = disease, linetype = ifelse(p < GW, "solid", "22")), linewidth = .3, alpha = .85) +
  geom_segment(data = direct, aes(sx, sy, xend = ex, yend = ey, colour = disease, linetype = lt), linewidth = .42,
               arrow = arrow(length = unit(1.5, "mm"), type = "closed", angle = 22)) +
  geom_segment(data = hubs, aes(hx, hy, xend = ex, yend = ey, colour = disease, linetype = lt), linewidth = .42,
               arrow = arrow(length = unit(1.5, "mm"), type = "closed", angle = 22)) +
  geom_circle(data = pos[!is.na(ring)], aes(x0 = x, y0 = y, r = ifelse(crispr, 2.5, 1.95), colour = ring, linetype = ring_lt), linewidth = .45,
              inherit.aes = FALSE, n = 60) +
  geom_polygon(data = gl, aes(x, y, group = id, fill = f), colour = NA) +
  geom_polygon(data = outl, aes(x, y, group = gene), fill = NA, colour = "grey20", linewidth = .25) +
  geom_label(data = pos, aes(tx, ty, label = gene, hjust = hj, vjust = vj, fontface = ifelse(crispr, "bold.italic", "italic")),
             size = 5 / .pt, fill = scales::alpha("white", .92), linewidth = 0, label.padding = unit(.08, "mm")) +
  geom_shape(data = dis[, .(px = c(x - w / 2, x + w / 2, x + w / 2, x - w / 2), py = c(y - 3.3, y - 3.3, y + 3.3, y + 3.3)), by = disease],
             aes(px, py, group = disease), fill = "white", colour = NA, radius = unit(1.2, "mm")) +
  geom_shape(data = dis[, .(px = c(x - w / 2, x + w / 2, x + w / 2, x - w / 2), py = c(y - 3.3, y - 3.3, y + 3.3, y + 3.3)), by = disease],
             aes(px, py, group = disease, fill = disease, colour = disease), radius = unit(1.2, "mm"), alpha = .07, linewidth = .35) +
  geom_text(data = dis, aes(x, y, label = sub(" ", "\n", disease)), size = 5 / .pt, lineheight = .9) +
  geom_segment(data = con, aes(x, y, xend = xend, yend = yend, colour = metaLV), linewidth = .35) +
  geom_shape(data = blkp, aes(x, y, group = metaLV, fill = metaLV, colour = metaLV), radius = unit(1.2, "mm"),
             alpha = .07, linewidth = .35) +
  geom_text(data = lab, aes(lx, ly, label = title), parse = TRUE, hjust = 0, vjust = 1, size = 6 / .pt) +
  geom_text(data = rows, aes(lx, y, label = k), hjust = 0, size = 5 / .pt, colour = "black") +
  geom_text(data = rows, aes(lx + KX, y, label = val), hjust = 0, size = 5 / .pt, colour = "black") +
  geom_rect(data = bars, aes(xmin = xmin, xmax = xmax, ymin = by1 - BH / 2, ymax = by1 + BH / 2, fill = category)) +
  geom_text(data = blab, aes(lx + KX + BW + .8, by1, label = act), hjust = 0, size = 5 / .pt) +
  scale_size_identity() +
  scale_fill_manual(values = c(setNames(sapply(pcol, function(c) colorRampPalette(c(c, "white"))(10)[8]), paste0(names(pcol), "_box")), ncol_, "MGM175+MGM335" = mix(ncol_[[1]], ncol_[[2]]), "MGM588+MGM91" = mix(ncol_[[3]], ncol_[[4]]), pcol, ccol), guide = "none") +
  scale_colour_manual(values = c(pcol, ncol_), guide = "none") +
  scale_linetype_identity() +
  coord_fixed(xlim = c(0, PANEL_W), ylim = c(0, PANEL_H), expand = FALSE, clip = "off") +
  theme_void() + theme(plot.margin = margin(0, 0, 0, 0))

leg_c <- data.table(category = factor(names(ccol), levels = names(ccol)), i = seq_along(ccol),
                    lab = c("liver / hepatocyte", "macrophage / monocyte", "pancreas / ductal", "blood / immune",
                            "brain / neural", "tumour / cancer", "plasma / cell-free", "other"))
leg_c[, `:=`(x = 66 + c(0, 23, 40)[(i - 1) %/% 3 + 1], y = 4.3 - (i - 1) %% 3 * 2.15)]
leg <- ggplot() +
  annotate("text", x = 0, y = 6.5, label = "Network", hjust = 0, size = 5.5 / .pt, fontface = "bold") +
  annotate("polygon", x = 1 + .9 * c(0, 1, 0, -1), y = 4.3 + .9 * c(1, 0, -1, 0), fill = "grey75", colour = "grey20", linewidth = .25) +
  annotate("text", x = 3.2, y = 4.3, label = "CRISPR screen gene", hjust = 0, size = 5 / .pt) +
  annotate("point", x = 1, y = 2.15, shape = 21, size = 1.9, fill = "grey75", colour = "grey20", stroke = .25) +
  annotate("text", x = 3.2, y = 2.15, label = "gene (top 10 loadings)", hjust = 0, size = 5 / .pt) +
  annotate("polygon", x = 1 + .65 * cos(seq(0, pi, length.out = 20)), y = 0 + .65 * sin(seq(0, pi, length.out = 20)), fill = ncol_[[1]]) +
  annotate("polygon", x = 1 + .65 * cos(seq(pi, 2 * pi, length.out = 20)), y = 0 + .65 * sin(seq(pi, 2 * pi, length.out = 20)), fill = ncol_[[2]]) +
  annotate("text", x = 3.2, y = 0, label = "gene shared by two MGMs", hjust = 0, size = 5 / .pt) +
  annotate("segment", x = 27.5, xend = 29.7, y = 4.3, yend = 4.3, colour = "grey60", linewidth = .2, linetype = "dashed") +
  annotate("text", x = 31.2, y = 4.3, label = "genes share an enriched pathway", hjust = 0, size = 5 / .pt) +
  annotate("point", x = 28.6, y = 2.15, shape = 21, size = 3.1, stroke = .55, colour = "grey30", fill = NA) +
  annotate("text", x = 31.2, y = 2.15, label = "S-MultiXcan FDR <0.05 (linked disease)", hjust = 0, size = 5 / .pt) +
  annotate("text", x = 66, y = 6.5, label = "Bars: top 1% ARCHS4 samples by module activity (B)", hjust = 0, size = 5.5 / .pt, fontface = "bold") +
  geom_tile(data = leg_c, aes(x + .8, y, fill = category), width = 1.6, height = 1.3) +
  geom_text(data = leg_c, aes(x + 2.3, y, label = lab), hjust = 0, size = 5 / .pt) +
  scale_fill_manual(values = ccol, guide = "none") +
  coord_fixed(xlim = c(0, PANEL_W), ylim = c(-1.2, 7.4), expand = FALSE, clip = "off") + theme_void()

## Panel b: single-gene TWAS and colocalization

In [ ]:
tlev <- unique(tw$trait); glev <- unique(tw$gene); screen <- unique(tw[screen == TRUE, gene])
tw[, `:=`(trait = factor(trait, levels = tlev), gene = factor(gene, levels = rev(glev)),
          nlp = pmin(-log10(pvalue), 60), rcp = fifelse(is.na(rcp), 0, rcp))]
tcol <- c(Gallstones = pcol[[1]], "Alzheimer's" = pcol[[2]], "High cholesterol" = pcol[[3]], "Venous thrombosis" = pcol[[4]])
tgroup <- unique(tw[, .(trait, trait_group)])[order(trait)]
plot_b <- ggplot(tw, aes(trait, gene)) +
  geom_tile(aes(fill = nlp), colour = "white", linewidth = .3) +
  geom_point(data = tw[rcp >= 0.01 & nlp < 8], aes(size = rcp), shape = 21, fill = NA, colour = "black", stroke = .3) +
  geom_point(data = tw[rcp >= 0.01 & nlp >= 8], aes(size = rcp), shape = 21, fill = NA, colour = "white", stroke = .3) +
  geom_point(data = tw[own == TRUE], shape = 23, size = 1.5, fill = NA, colour = "black", stroke = .3) +
  scale_fill_gradientn(colours = c("#F6F8F7", "#EEF2F0", "#9ACDB8", "#4E9E81", "#2A6E58", "#1B4D43"),
                       values = scales::rescale(c(0, 1.3, 3.5, 8, 20, 60), from = c(0, 60)), limits = c(0, 60),
                       breaks = c(0, 20, 40, 60),
                       name = "S-MultiXcan\n-log10(P), capped at 60") +
  scale_size_area(max_size = 2.6, limits = c(0, 0.6), breaks = c(0.1, 0.3, 0.5), name = "RCP") +
  scale_x_discrete(position = "top") + labs(x = NULL, y = NULL) +
  theme_classic(base_size = 5, base_family = "Helvetica") +
  theme(axis.line = element_blank(), axis.ticks = element_blank(), axis.text = element_text(size = 5, colour = "black"),
        axis.text.x = element_text(angle = 55, hjust = 0, colour = tcol[tgroup$trait_group]),
        axis.text.y = element_text(face = ifelse(levels(tw$gene) %in% screen, "bold.italic", "italic")),
        legend.text = element_text(size = 5), legend.title = element_text(size = 5.5),
        legend.position = "bottom", legend.box = "vertical", legend.margin = margin(0, 0, 0, 0),
        legend.key.width = unit(4, "mm"), legend.key.height = unit(1.4, "mm"), legend.title.position = "top",
        plot.margin = margin(1.3, 10, 1.3, 1.3)) +
  guides(fill = guide_colourbar(order = 1), size = guide_legend(order = 2, nrow = 1))

## Assemble and save

In [ ]:
tag <- function(p, l) ggdraw() + draw_plot(p, x = .015, y = 0, width = .985, height = .97) +
  draw_label(l, x = .002, y = .998, hjust = 0, vjust = 1, size = FS_TAG, fontface = "bold", fontfamily = "Helvetica")
pa <- plot_grid(plot_a, leg, ncol = 1, rel_heights = c(PANEL_H, 9.5))
fig <- plot_grid(tag(pa, "a"), tag(plot_b, "b"), nrow = 1, rel_widths = c(PANEL_W + 4, FIG_W - PANEL_W - 4))
for (ext in c("pdf", "png", "svg")) {
  dev <- switch(ext, pdf = cairo_pdf, png = ragg::agg_png, svg = svg_device)
  ggsave(file.path(dirs$out, paste0("fig5.", ext)), fig, width = FIG_W, height = FIG_H, units = "mm", device = dev, bg = "white", dpi = 400)
  if (ext == "svg") add_svg_font_stack(file.path(dirs$out, "fig5.svg"))
}
options(repr.plot.width = FIG_W / 25.4, repr.plot.height = FIG_H / 25.4)
print(fig)